In [1]:
# ============================================
# 📊 Memory Measurement for SDXL Lightning FP16
# Purpose: Measure GPU/MPS memory usage during inference
# ============================================

import torch
import gc
import os
import psutil
import time
from pathlib import Path
from diffusers import StableDiffusionXLPipeline
import json

# ============================================
# ⚙️ Configuration
# ============================================

class MemoryConfig:
    MODEL_PATH = "./sdxl/quantized_models/sdxl_lightning_4step_mps_fp16"
    
    # Test settings
    IMAGE_SIZE = 512
    NUM_INFERENCE_STEPS = 4
    GUIDANCE_SCALE = 0.0
    
    # Test prompts (different lengths to test various scenarios)
    TEST_PROMPTS = [
        "A cat",  # Short
        "A serene mountain landscape at sunset",  # Medium
        "A highly detailed photorealistic portrait of an astronaut exploring an alien planet with bioluminescent plants",  # Long
    ]
    
    # Output
    RESULTS_FILE = Path("./evaluation_outputs/memory_measurement_results.json")

config = MemoryConfig()
config.RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)

# Device detection
if torch.backends.mps.is_available():
    device = "mps"
    print("✅ Using Apple Silicon (MPS)")
elif torch.cuda.is_available():
    device = "cuda"
    print("✅ Using CUDA GPU")
else:
    device = "cpu"
    print("⚠️  Using CPU (no GPU detected)")


# ============================================
# 🧹 Memory Utilities
# ============================================

def clear_memory():
    """Aggressively clear all memory caches."""
    gc.collect()
    if device == "mps":
        torch.mps.empty_cache()
    elif device == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def get_memory_stats():
    """Get current memory usage statistics."""
    stats = {
        "ram_used_gb": psutil.Process().memory_info().rss / (1024**3),
        "ram_percent": psutil.virtual_memory().percent,
    }
    
    if device == "mps":
        # MPS memory (Apple Silicon)
        stats["mps_allocated_gb"] = torch.mps.current_allocated_memory() / (1024**3)
        stats["mps_reserved_gb"] = torch.mps.driver_allocated_memory() / (1024**3)
    elif device == "cuda":
        # CUDA memory
        stats["gpu_allocated_gb"] = torch.cuda.memory_allocated() / (1024**3)
        stats["gpu_reserved_gb"] = torch.cuda.memory_reserved() / (1024**3)
        stats["gpu_max_allocated_gb"] = torch.cuda.max_memory_allocated() / (1024**3)
    
    return stats


def print_memory_stats(stats, label=""):
    """Pretty print memory statistics."""
    print(f"\n{'='*50}")
    print(f"💾 Memory Stats: {label}")
    print(f"{'='*50}")
    print(f"RAM Used: {stats['ram_used_gb']:.2f} GB ({stats['ram_percent']:.1f}%)")
    
    if device == "mps":
        print(f"MPS Allocated: {stats['mps_allocated_gb']:.2f} GB")
        print(f"MPS Reserved: {stats['mps_reserved_gb']:.2f} GB")
    elif device == "cuda":
        print(f"GPU Allocated: {stats['gpu_allocated_gb']:.2f} GB")
        print(f"GPU Reserved: {stats['gpu_reserved_gb']:.2f} GB")
        print(f"GPU Peak: {stats['gpu_max_allocated_gb']:.2f} GB")


# ============================================
# 📏 Memory Measurement Functions
# ============================================

def measure_model_loading_memory():
    """Measure memory during model loading."""
    print("\n" + "="*70)
    print("🔧 PHASE 1: Model Loading Memory")
    print("="*70)
    
    clear_memory()
    baseline = get_memory_stats()
    print_memory_stats(baseline, "Before Loading Model")
    
    print("\n⏳ Loading model...")
    start_time = time.time()
    
    pipe = StableDiffusionXLPipeline.from_pretrained(
        config.MODEL_PATH,
        torch_dtype=torch.float16,
        use_safetensors=True,
    )
    pipe = pipe.to(device)
    
    load_time = time.time() - start_time
    
    after_load = get_memory_stats()
    print_memory_stats(after_load, "After Loading Model")
    
    # Calculate memory increase
    if device == "mps":
        memory_increase = after_load['mps_allocated_gb'] - baseline['mps_allocated_gb']
    elif device == "cuda":
        memory_increase = after_load['gpu_allocated_gb'] - baseline['gpu_allocated_gb']
    else:
        memory_increase = after_load['ram_used_gb'] - baseline['ram_used_gb']
    
    print(f"\n📈 Model Size in Memory: {memory_increase:.2f} GB")
    print(f"⏱️  Load Time: {load_time:.2f}s")
    
    return pipe, {
        "baseline": baseline,
        "after_load": after_load,
        "memory_increase_gb": memory_increase,
        "load_time_seconds": load_time,
    }


def measure_inference_memory(pipe, prompt, image_size=512):
    """Measure memory during a single inference."""
    clear_memory()
    before_inference = get_memory_stats()
    
    print(f"\n⏳ Generating image (prompt length: {len(prompt)} chars)...")
    
    generator = torch.Generator(device="cpu").manual_seed(42)
    start_time = time.time()
    
    # Run inference
    image = pipe(
        prompt=prompt,
        num_inference_steps=config.NUM_INFERENCE_STEPS,
        guidance_scale=config.GUIDANCE_SCALE,
        generator=generator,
        height=image_size,
        width=image_size,
    ).images[0]
    
    inference_time = time.time() - start_time
    
    after_inference = get_memory_stats()
    
    # Memory delta during inference
    if device == "mps":
        peak_memory = after_inference['mps_allocated_gb']
        inference_overhead = peak_memory - before_inference['mps_allocated_gb']
    elif device == "cuda":
        peak_memory = after_inference['gpu_max_allocated_gb']
        inference_overhead = peak_memory - before_inference['gpu_allocated_gb']
    else:
        peak_memory = after_inference['ram_used_gb']
        inference_overhead = peak_memory - before_inference['ram_used_gb']
    
    print(f"  ✅ Done in {inference_time:.2f}s")
    print(f"  📊 Inference overhead: {inference_overhead:.3f} GB")
    
    return {
        "prompt_length": len(prompt),
        "inference_time_seconds": inference_time,
        "before": before_inference,
        "after": after_inference,
        "peak_memory_gb": peak_memory,
        "inference_overhead_gb": inference_overhead,
    }


def measure_batch_inference_memory(pipe, prompts):
    """Measure memory across multiple prompts."""
    print("\n" + "="*70)
    print("🔧 PHASE 2: Inference Memory Measurements")
    print("="*70)
    
    results = []
    
    for i, prompt in enumerate(prompts, 1):
        print(f"\n📝 Test {i}/{len(prompts)}: \"{prompt[:50]}...\"")
        result = measure_inference_memory(pipe, prompt, config.IMAGE_SIZE)
        results.append(result)
        
        # Brief pause between tests
        time.sleep(1)
    
    return results


def measure_memory_with_optimizations(pipe):
    """Compare memory usage with different optimizations."""
    print("\n" + "="*70)
    print("🔧 PHASE 3: Memory Optimization Comparisons")
    print("="*70)
    
    test_prompt = config.TEST_PROMPTS[1]  # Use medium-length prompt
    optimizations = {}
    
    # Test 1: No optimizations (baseline)
    print("\n🧪 Test 1: No optimizations")
    clear_memory()
    result = measure_inference_memory(pipe, test_prompt)
    optimizations["no_optimization"] = result
    
    # Test 2: Attention slicing
    print("\n🧪 Test 2: With attention slicing")
    pipe.enable_attention_slicing(slice_size="auto")
    clear_memory()
    result = measure_inference_memory(pipe, test_prompt)
    optimizations["attention_slicing"] = result
    pipe.disable_attention_slicing()
    
    # Test 3: VAE slicing
    print("\n🧪 Test 3: With VAE slicing")
    pipe.enable_vae_slicing()
    clear_memory()
    result = measure_inference_memory(pipe, test_prompt)
    optimizations["vae_slicing"] = result
    pipe.disable_vae_slicing()
    
    # Test 4: Both optimizations
    print("\n🧪 Test 4: With both optimizations")
    pipe.enable_attention_slicing(slice_size="auto")
    pipe.enable_vae_slicing()
    clear_memory()
    result = measure_inference_memory(pipe, test_prompt)
    optimizations["both_optimizations"] = result
    
    return optimizations


# ============================================
# 📊 Results Summary
# ============================================

def summarize_results(loading_results, inference_results, optimization_results):
    """Create a comprehensive summary of all measurements."""
    
    # Calculate averages for inference
    avg_inference_time = sum(r['inference_time_seconds'] for r in inference_results) / len(inference_results)
    avg_inference_overhead = sum(r['inference_overhead_gb'] for r in inference_results) / len(inference_results)
    max_peak_memory = max(r['peak_memory_gb'] for r in inference_results)
    
    summary = {
        "model_info": {
            "model_path": config.MODEL_PATH,
            "device": device,
            "precision": "float16",
        },
        "model_loading": {
            "model_size_in_memory_gb": loading_results['memory_increase_gb'],
            "load_time_seconds": loading_results['load_time_seconds'],
        },
        "inference": {
            "image_size": config.IMAGE_SIZE,
            "num_steps": config.NUM_INFERENCE_STEPS,
            "avg_inference_time_seconds": avg_inference_time,
            "avg_inference_overhead_gb": avg_inference_overhead,
            "max_peak_memory_gb": max_peak_memory,
            "total_memory_required_gb": loading_results['memory_increase_gb'] + avg_inference_overhead,
        },
        "optimizations": {
            name: {
                "inference_time_seconds": data['inference_time_seconds'],
                "peak_memory_gb": data['peak_memory_gb'],
                "inference_overhead_gb": data['inference_overhead_gb'],
            }
            for name, data in optimization_results.items()
        },
        "detailed_inference_results": inference_results,
    }
    
    return summary


def print_final_summary(summary):
    """Print a nice final summary."""
    print("\n" + "="*70)
    print("📋 FINAL MEMORY MEASUREMENT SUMMARY")
    print("="*70)
    
    print(f"\n🔧 Model Configuration:")
    print(f"   Device: {summary['model_info']['device'].upper()}")
    print(f"   Precision: {summary['model_info']['precision']}")
    print(f"   Model: {Path(summary['model_info']['model_path']).name}")
    
    print(f"\n💾 Model Loading:")
    print(f"   Model size in memory: {summary['model_loading']['model_size_in_memory_gb']:.2f} GB")
    print(f"   Load time: {summary['model_loading']['load_time_seconds']:.2f}s")
    
    print(f"\n🖼️  Inference ({config.IMAGE_SIZE}x{config.IMAGE_SIZE}, {config.NUM_INFERENCE_STEPS} steps):")
    print(f"   Average inference time: {summary['inference']['avg_inference_time_seconds']:.2f}s")
    print(f"   Average inference overhead: {summary['inference']['avg_inference_overhead_gb']:.3f} GB")
    print(f"   Peak memory usage: {summary['inference']['max_peak_memory_gb']:.2f} GB")
    print(f"   Total memory required: {summary['inference']['total_memory_required_gb']:.2f} GB")
    
    print(f"\n⚡ Memory Optimization Comparison:")
    baseline = summary['optimizations']['no_optimization']['peak_memory_gb']
    for name, data in summary['optimizations'].items():
        savings = baseline - data['peak_memory_gb']
        savings_pct = (savings / baseline) * 100
        print(f"   {name:25s}: {data['peak_memory_gb']:.2f} GB (saves {savings:.2f} GB / {savings_pct:.1f}%)")
    
    print(f"\n💡 Recommendations:")
    if summary['inference']['total_memory_required_gb'] > 16:
        print("   ⚠️  Model requires >16GB - consider using optimizations")
    elif summary['inference']['total_memory_required_gb'] > 12:
        print("   ⚠️  Model requires >12GB - may struggle on some devices")
    else:
        print("   ✅ Memory usage is reasonable for most devices")
    
    best_optimization = min(summary['optimizations'].items(), 
                           key=lambda x: x[1]['peak_memory_gb'])
    print(f"   🏆 Best optimization: {best_optimization[0]} ({best_optimization[1]['peak_memory_gb']:.2f} GB)")


# ============================================
# 🚀 Main Execution
# ============================================

def run_full_memory_analysis():
    """Run complete memory measurement analysis."""
    
    print("="*70)
    print("🚀 Starting Full Memory Analysis")
    print("="*70)
    
    # Phase 1: Model loading
    pipe, loading_results = measure_model_loading_memory()
    
    # Phase 2: Inference measurements
    inference_results = measure_batch_inference_memory(pipe, config.TEST_PROMPTS)
    
    # Phase 3: Optimization comparisons
    optimization_results = measure_memory_with_optimizations(pipe)
    
    # Generate summary
    summary = summarize_results(loading_results, inference_results, optimization_results)
    
    # Print summary
    print_final_summary(summary)
    
    # Save results
    with open(config.RESULTS_FILE, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"\n💾 Full results saved to: {config.RESULTS_FILE}")
    
    # Cleanup
    del pipe
    clear_memory()
    
    return summary


# Run the analysis
if __name__ == "__main__":
    results = run_full_memory_analysis()

/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Using Apple Silicon (MPS)
🚀 Starting Full Memory Analysis

🔧 PHASE 1: Model Loading Memory

💾 Memory Stats: Before Loading Model
RAM Used: 0.36 GB (81.9%)
MPS Allocated: 0.00 GB
MPS Reserved: 0.00 GB

⏳ Loading model...


Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00,  8.29it/s]



💾 Memory Stats: After Loading Model
RAM Used: 0.02 GB (72.7%)
MPS Allocated: 6.46 GB
MPS Reserved: 7.12 GB

📈 Model Size in Memory: 6.46 GB
⏱️  Load Time: 27.33s

🔧 PHASE 2: Inference Memory Measurements

📝 Test 1/3: "A cat..."

⏳ Generating image (prompt length: 5 chars)...


100%|██████████| 4/4 [00:05<00:00,  1.36s/it]


...

  ✅ Done in 6.83s
  📊 Inference overhead: 0.002 GB

🧪 Test 3: With VAE slicing

⏳ Generating image (prompt length: 37 chars)...


100%|██████████| 4/4 [00:03<00:00,  1.33it/s]


  ✅ Done in 4.82s
  📊 Inference overhead: 0.000 GB

🧪 Test 4: With both optimizations

⏳ Generating image (prompt length: 37 chars)...


100%|██████████| 4/4 [00:02<00:00,  1.55it/s]


  ✅ Done in 4.43s
  📊 Inference overhead: 0.001 GB

📋 FINAL MEMORY MEASUREMENT SUMMARY

🔧 Model Configuration:
   Device: MPS
   Precision: float16
   Model: sdxl_lightning_4step_mps_fp16

💾 Model Loading:
   Model size in memory: 6.46 GB
   Load time: 27.33s

🖼️  Inference (512x512, 4 steps):
   Average inference time: 8.39s
   Average inference overhead: 0.004 GB
   Peak memory usage: 6.47 GB
   Total memory required: 6.47 GB

⚡ Memory Optimization Comparison:
   no_optimization          : 6.48 GB (saves 0.00 GB / 0.0%)
   attention_slicing        : 6.48 GB (saves -0.00 GB / -0.0%)
   vae_slicing              : 6.48 GB (saves -0.00 GB / -0.0%)
   both_optimizations       : 6.48 GB (saves -0.00 GB / -0.0%)

💡 Recommendations:
   ✅ Memory usage is reasonable for most devices
   🏆 Best optimization: no_optimization (6.48 GB)

💾 Full results saved to: evaluation_outputs/memory_measurement_results.json
